In [2]:
import os
import json
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import operator
import pandas as pd
import copy

각 알약 조합 별, 모든 알약에 대해 정보가 존재하는 이미지들을 구한다.

In [2]:
images=sorted(os.listdir('./ai06-level1-project/train_output'))
annot_root='./ai06-level1-project/train_annotations'
annots_t=sorted(os.listdir(annot_root)) # 알약 조합 목록
annots_list=[]
images_list=[]
# 각 알약 조합에 대해
for annot_set in annots_t:
    # 약 목록을 구한다.
    pill_list=sorted(os.listdir(os.path.join(annot_root,annot_set)))
    # 첫 약이 가지는 json 파일 목록 (각 이미지에 대한 json 파일 목록)
    base_images=sorted(os.listdir(os.path.join(annot_root,annot_set,pill_list[0])))
    # 각 약에 대해
    for pill in pill_list:
        # 각 약이 어떤 이미지에 대한 annotation을 갖는지에 대한 list
        imgs=sorted(os.listdir(os.path.join(annot_root,annot_set,pill)))
        # 특정 약은 annotation이 없는 이미지여도 추가
        for img in imgs:
            if img not in base_images:
                base_images.append(img)
    # 모든 약에 대해, 공통으로 json 파일이 있는 이미지의 이름만 저장
    for image in base_images:
        annots_list.append(image[:-5])
# 
for img in images:
    images_list.append(img[:-4])
len(annots_list), annots_list[0], len(images_list),images_list[0]

(371,
 'K-001900-016548-019607-029451_0_2_0_2_70_000_200',
 651,
 'K-001900-010224-016551-031705_0_2_0_2_70_000_200')

전체 파일 중, annotation이 하나라도 있는 이미지들의 이름만 추출

In [3]:
images_set=set(images_list)
annots_set=set(annots_list)
# 이미지에 대해, annotation이 있는 경우의 이름들을 모음
available_files=sorted(list(images_set.intersection(annots_set)))
len(available_files)

234

In [4]:
for f in available_files:
    print(f)
    break

K-001900-016548-019607-029451_0_2_0_2_70_000_200


In [5]:
pill_count={}

이미지 단위로 annotation 파일들을 결합합니다.</br>
이때, 추후 추가 데이터를 사용하기 위해서 image의 id 목록과 annot의 id 목록을 따로 얻습니다.

In [6]:
annot_ids=[]
image_ids=[]

In [7]:
#json 파일 통합

base_path = "./ai06-level1-project"
ann_root = os.path.join(base_path, "train_annotations")

output_path = os.path.join(base_path, "train_annots")
os.makedirs(output_path, exist_ok=True)

image_id_offset = 0
annotation_id_offset = 0
category_set = set()

# 알약 조합 별 폴더 명
for root in sorted(os.listdir(ann_root)):

    # 현재 조합의 알약 종류
    pills=sorted(os.listdir(os.path.join(ann_root,root)))

    # 현재 조합의 이미지 목록
    images_list=[]
    for pill in pills:
        images_list_p=os.listdir(os.path.join(ann_root,root,pill))
        for i in images_list_p:
            if i not in images_list:
                images_list.append(i)

    if len(images_list)==0:
        continue

    # 각 이미지 별 JSON 파일 생성
    for img in images_list:

        images=[]
        annotations=[]
        categories=[]

        # 대응하는 이미지가 없거나, 일부 알약에 대해 누락되어 있다 annotation이 누락된 이미지면 continue
        if img[:-5] not in available_files:
            continue
        # 각 알약에 대해서
        for pill in pills:
            # 만약 이번 알약은 해당 이미지에 대한 annotation이 없다면 넘어갑니다.
            if img not in os.listdir(os.path.join(ann_root,root,pill)):
                continue
            # 해당 이미지의 json 파일을 엽니다.
            with open(str(os.path.join(ann_root,root,pill,img)), "r", encoding="utf-8") as f:
                pill_data = json.load(f)
                if len(images)==0:
                    images.extend(pill_data["images"])
                    image_ids.append(pill_data['images'][0]['id'])
                annotations.extend(pill_data["annotations"])
                annot_ids.append(pill_data['annotations'][0]['id'])
                categories.extend(pill_data["categories"])
                cat= pill_data['categories'][0]
                if cat['id'] not in pill_count.keys():
                    pill_count[cat['id']]=1
                else:
                    pill_count[cat['id']]+=1
        output_coco={
            "images": images,
            "annotations": annotations,
            "categories": categories
        }
        with open(os.path.join(output_path,img), "w", encoding="utf-8") as f:
            json.dump(output_coco, f, ensure_ascii=False, indent=4)

In [8]:
pill_count={}

이제 모든 이미지의 json 파일을 결합합니다.

In [9]:
output_all_coco = {
    "images": [],
    "annotations": [],
    "categories": []
}
files = os.listdir(os.path.join(base_path, "train_annots"))

for file in files:
    with open(os.path.join(base_path, "train_annots", file), "r", encoding="utf-8") as f:
        data=json.load(f)
        for image in data['images']:
            output_all_coco['images'].append(image)
        for annot in data['annotations']:
            output_all_coco['annotations'].append(annot)
        for cat in data['categories']:
            print(cat)
            output_all_coco['categories'].append(cat)

t=pd.DataFrame(output_all_coco['categories'])
t=t.drop_duplicates()
output_all_coco['categories']=t.to_dict(orient='records')

save_path = os.path.join(base_path, "train.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output_all_coco, f, ensure_ascii=False, indent=4)

print("train.json 생성됨:", save_path)

{'supercategory': 'pill', 'id': 1899, 'name': '보령부스파정 5mg'}
{'supercategory': 'pill', 'id': 16547, 'name': '가바토파정 100mg'}
{'supercategory': 'pill', 'id': 19606, 'name': '스토가정 10mg'}
{'supercategory': 'pill', 'id': 29450, 'name': '레일라정'}
{'supercategory': 'pill', 'id': 1899, 'name': '보령부스파정 5mg'}
{'supercategory': 'pill', 'id': 16547, 'name': '가바토파정 100mg'}
{'supercategory': 'pill', 'id': 19606, 'name': '스토가정 10mg'}
{'supercategory': 'pill', 'id': 29450, 'name': '레일라정'}
{'supercategory': 'pill', 'id': 1899, 'name': '보령부스파정 5mg'}
{'supercategory': 'pill', 'id': 16547, 'name': '가바토파정 100mg'}
{'supercategory': 'pill', 'id': 19606, 'name': '스토가정 10mg'}
{'supercategory': 'pill', 'id': 29450, 'name': '레일라정'}
{'supercategory': 'pill', 'id': 1899, 'name': '보령부스파정 5mg'}
{'supercategory': 'pill', 'id': 16547, 'name': '가바토파정 100mg'}
{'supercategory': 'pill', 'id': 19606, 'name': '스토가정 10mg'}
{'supercategory': 'pill', 'id': 33008, 'name': '신바로정'}
{'supercategory': 'pill', 'id': 1899, 'name': '보령부스파

In [10]:
pill_code=[]
annots_t=sorted(os.listdir('./ai06-level1-project/train_annotations')) # 알약 조합 목록
for annot_set in annots_t:
    pill_list=sorted(os.listdir(os.path.join('./ai06-level1-project/train_annotations',annot_set)))
    for pill in pill_list:
        if int(pill[2:])-1 not in pill_code:
            t=os.path.join('./ai06-level1-project/train_annotations',annot_set,pill)
            a=os.listdir(t)[0]
            with open(os.path.join(t,a)) as f:
                j=int(json.load(f)['images'][0]['dl_idx'])
                pill_code.append(j)

알약 카테고리 별 수를 구한다.

In [11]:
count=sorted(pill_count.items(),key=operator.itemgetter(1),reverse=True)
with open('./pill_count.txt','w') as f:
    for i in count:
        if i[0] in pill_code:
            f.write(f"{i[0]}\t{i[1]}\n")

In [12]:
def PIL2OpenCV(pil_image):
    numpy_image= np.array(pil_image)
    opencv_image = cv2.cvtColor(numpy_image, cv2.COLOR_RGB2BGR)
    return opencv_image

def OpenCV2PIL(opencv_image):
    color_coverted = cv2.cvtColor(opencv_image, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(color_coverted)
    return pil_image

In [ ]:
images=sorted(os.listdir('./ai06-level1-project/train_output'))
annotations=sorted(os.listdir('./ai06-level1-project/train_annotations'))

for image in images:
    # 알약 조합은 이름에서 확장자 및 이미지 정보를 제거한 것과 같다.
    name=image[:-23]

    # 만약 annotation에 해당 알약 조합의 정보가 없다면, 폴더를 만들어준다.
    if not os.path.isdir('./ai06-level1-project/train_annotations/'+name+"_json"):
        os.makedirs('./ai06-level1-project/train_annotations/'+name+"_json")

    # 이미지 파일을 불러온다.
    image_file=Image.open(os.path.join('./ai06-level1-project/train_output',image))

    # 이미지 파일을 cv 파일로, 배경을 유지하여 만들어준다.
    image_cv=cv2.cvtColor(np.array(image_file), cv2.COLOR_RGBA2BGRA)

    # 이미지 파일을 cv 파일로, 배경을 지워서 만들어준다.
    image_file=PIL2OpenCV(image_file.convert("RGB"))

    # 이미지 파일에 변형을 가할 파일을 만들어준다.
    conv_file=copy.deepcopy(image_file)

    # 이미지 파일 원본의 크기를 구한다.
    H,W,C=image_file.shape

    # 각 칸에 대해, 배경에 해당하면 흰색, 아니면 검은 색으로 만들어준다.
    for h in range(H):
        for w in range(W):
            if list(image_file[h,w])==[130,130,130]:
                for c in range(C):
                    conv_file[h,w]=255
            else:
                for c in range(C):
                    conv_file[h,w]=0
    
    # 검출된 3~4개의 이미지를 저장할 배열이다.
    imgs=[]

    # 행 단위로 (horizontal) 직전 행의 모든 칸이 흰색인지 체크한다.
    check_h=True
    h_line=[]
    for h in range(H):
        # 이번 행에서 모든 칸이 흰색인가에 대해서
        cur_row=(conv_file[h]==255).all()
        # 만약 cur_row와 check_h가 서로 다르다면 h_line에 추가
        if check_h != cur_row:
            h_line.append(h)
            check_h=cur_row
    # 행 단위로 나누는 선을 만든다.
    h_line=[h_line[0]//2,(h_line[1]+h_line[2])//2,((h_line[3]+H)//2)]

    # 행 단위로 2개의 파트를 만든다.
    conv1=conv_file[h_line[0]:h_line[1]]
    conv2=conv_file[h_line[1]:h_line[2]]

    # 파트 1의 이미지 수를 구한다.
    check_v=True
    v_line1=[]
    for w in range(W):
        # 이번 열에서 파트 1의 모든 칸이 흰색인가에 대해서
        cur_col=(conv1[:,w]==255).all()
        # 만약 cur_col과 check_v가 서로 다르다면 v_line1에 추가
        if check_v != cur_col:
            v_line1.append(w)
            check_v=cur_col
    # v_line1의 길이가 4라면, imgs에 2개의 이미지를 넣어준다.
    if len(v_line1)==4:
        v_line1=[v_line1[0]//2,(v_line1[1]+v_line1[2])//2,((v_line1[3]+W)//2)]
        imgs.append(image_cv[h_line[0]:h_line[1],v_line1[0]:v_line1[1]])
        imgs.append(image_cv[h_line[0]:h_line[1],v_line1[1]:v_line1[2]])
    # 아니라면 imgs에는 1개의 이미지가 추가된다.
    else:
        v_center=sum(v_line1)//2
        v_line1=[v_center-256,v_center+256]
        imgs.append(image_cv[h_line[0]:h_line[1],v_line1[0]:v_line1[1]])

    # 파트 2에 대해서도 마찬가지로 한다.
    check_v=True
    v_line2=[]
    for w in range(W):        
        cur_col=(conv1[:,w]==255).all()
        # 만약 cur_col과 check_v가 서로 다르다면 v_line2에 추가
        if check_v != cur_col:
            v_line2.append(w)
            check_v=cur_col
    # v_line2의 길이가 4라면, imgs에 2개의 이미지를 넣어준다.
    if len(v_line2)==4:
        v_line2=[v_line2[0]//2,(v_line2[1]+v_line2[2])//2,((v_line2[3]+W)//2)]
        imgs.append(image_cv[h_line[0]:h_line[1],v_line2[0]:v_line2[1]])
        imgs.append(image_cv[h_line[0]:h_line[1],v_line2[1]:v_line2[2]])
    # 아니라면 imgs에는 1개의 이미지가 추가된다.
    else:
        v_center=sum(v_line2)//2
        v_line2=[v_center-256,v_center+256]
        imgs.append(image_cv[h_line[0]:h_line[1],v_line2[0]:v_line2[1]])

    # imgs의 각 이미지를 폴더에 저장해준다.
    for idx,img in enumerate(imgs):
        image_i=Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA))
        image_i.save(f"./cropped_images/{image[:-4]}_{str(idx+1)}.png", format="PNG")

    # 이미지 정보를 txt로 저장한다.
    with open(f'./ai06-level1-project/train_annotations/{name}_json/{image[:-4]}.txt','w') as f:
        f.write(f'H: {H}\n')
        f.write(f'W: {W}\n')
        cnt=0
        # 만약 파트 1의 이미지가 2개라면
        if len(v_line1)==3:
            f.write(f'img{str(cnt+1)}: {h_line[0]},{h_line[1]},{v_line1[0]},{v_line1[1]}\n')
            f.write(f'img{str(cnt+2)}: {h_line[0]},{h_line[1]},{v_line1[1]},{v_line1[2]}\n')
            cnt+=2
        else:
            f.write(f'img{str(cnt+1)}: {h_line[0]},{h_line[1]},{v_line1[0]},{v_line1[1]}\n')
            cnt+=1
        # 만약 파트 2의 이미지가 2개라면
        if len(v_line2)==3:
            f.write(f'img{str(cnt+1)}: {h_line[1]},{h_line[2]},{v_line2[0]},{v_line2[1]}\n')
            f.write(f'img{str(cnt+2)}: {h_line[1]},{h_line[2]},{v_line2[1]},{v_line2[2]}\n')
            cnt+=2
        else:
            f.write(f'img{str(cnt+1)}: {h_line[1]},{h_line[2]},{v_line2[0]},{v_line2[1]}\n')
            cnt+=1

이제, 각 이미지 정보 폴더 (annotation 폴더)들을 돌아다니면서 하위의 3~4개 이미지마다 상세 bbox를 구한다.</br>
그 뒤, 각 이미지마다 상세 bbox를 활용해 잘린 이미지를 저장해준다.</br>
상세 bbox는 추후 YOLO 등의 모델 학습 용도로 사용 가능하다.

In [ ]:
# 각 이미지 정보 폴더를 순회한다.
annotation_root='./ai06-level1-project/train_annotations'
annotations=sorted(os.listdir(annotation_root))

for annotation in annotations:
    # txt 파일들 목록을 불러온다.
    image_info=[entry.name for entry in os.scandir(os.path.join(annotation_root,annotation)) if entry.is_file() and entry.name.endswith(".txt")]
    # 이미지 하나에 대한 txt 파일마다
    for sub_info in image_info:
        with open(os.path.join(annotation_root,annotation,sub_info),'r') as f:
            # 우선 이미지의 높이, 너비 정보를 얻는다.
            H=f.readline().strip()[3:]
            W=f.readline().strip()[3:]
            print(H,W)
            parts=f.readlines()
            for part in parts:
                print(part)
        break
    break

각 상세 이미지를 활용해서 라벨링 데이터를 먼저 만들어주고, 이후 각 상세 이미지에 **투명 배경**을 패딩으로 추가하여 분류기를 학습시킨다.</br>
추후 예측 시에는
- 각 알약마다 bbox 위치를 3~4개 구해준 뒤
- 각 위치마다 분류기에 넣어 예측하고
` 그 결과를 출력물에 넣는다.